# ROSP-23: Inter-visit interval breakdown

Diagnostic notebook for PTH Tier 1 metric **ROSP-10** (open-shutter fraction and slew/settle time).

- Jira epic: [ROSP-10](https://rubinobs.atlassian.net/browse/ROSP-10)
- Jira ticket: [ROSP-23](https://rubinobs.atlassian.net/browse/ROSP-23)
- Performance Tracking Hub: [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)

| | |
|---|---|
| Steward | Erik Dennihy |
| Owner | Sandrine Thomas |
| Category | 1 - Observatory Operations |
| Parent metric | ROSP-10 (Layer 3) |

Local diagnostic notebook, not published on Times Square.

## Goal

This notebook presents a breakdown of the inter-visit interval into **thirteen serial segments**. Each segment is bounded by EFD events, has one mechanism and one owner, and the thirteen sum exactly to every gap. It relies on Observatory State updates to guard against long gaps caused by Faults, so it may return spurious results for data taken before 2026-02, when the observatory state features were deployed. 

The **visit gap** (OSS-REQ-0289) runs from the shutter close of one visit to the shutter open of the next. Each visit is a single exposure, so:

$$\text{visit gap} = t_{\texttt{startShutterOpen}}(\text{visit } N{+}1) - t_{\texttt{endShutterClose}}(\text{visit } N)$$

All times come from the EFD. The ConsDB is used only to identify the survey visits.

### The thirteen segments

| # | Segment | What it covers | Owner |
|---|---|---|---|
| 1 | Script startup | script N ends, script N+1 starts running | ScriptQueue |
| 2 | Script & TCS pre-slew checks | feasibility and MTCS entry checks | Scripts / MTCS |
| 3 | Telescope prep (M1M3) | booster valves open before the slew | M1M3 |
| 4 | AOS Correction (pre-slew) | applying a correction that was already waiting | AOS |
| 5 | Slew start latency | slew command until the first axis moves | TMA / pointing |
| 6 | Mount slew+settle | axis motion and settling until both axes are in position | TMA |
| 7 | Rotator catch-up | rotator time beyond the mount | Rotator |
| 8 | Dome catch-up | dome time beyond mount + rotator | Dome |
| 9 | In-position checks | MTCS confirming everything is in position | OCS / MTCS |
| 10 | AOS Correction (post-slew) | applying a correction that arrived after the slew started | AOS |
| 11 | AOS pointing correction | re-pointing after that correction | AOS / TMA |
| 12 | Camera dispatch | handing the exposure to the camera (after a post-slew correction) | OCS / Camera |
| 13 | Camera start | `takeImages` until the shutter opens | Camera / CCS |

Segments 4 and 10–12 are zero when there is no AOS correction. Segments 7–8 are usually zero. Filter-change gaps are not decomposed and are reported separately.

## Contents

1. **Data sources**
2. **Selection:** which gaps count
3. **Computation:** segment boundaries and the decomposition
4. **Deep dive:** example gaps, event by event
5. **Statistics:** per-segment median, mean, min and max
6. **Night-to-night comparison** (ranges only)
7. **Results:** headline and ranked contributors

To run one night, set `day_obs` and leave `day_obs_end = None`. To run a range, set both.

In [ ]:
# Parameters.
day_obs = 20260629  # night to analyse, or first night of a range (YYYYMMDD)
day_obs_end = 20261231  # last night of a range (inclusive); None = single night
inspect_visit_id = (
    None  # draw the gaps on either side of this visit (e.g. 2026071300493); None = skip
)
verbose_output = False  # True: also show diagnostic prints and tables

In [ ]:
# Validate the parameters and build the list of nights.
# vprint / vdisplay print only when verbose_output is True.
from datetime import datetime, timedelta

from IPython.display import display

day_obs = int(day_obs)
day_obs_end = day_obs if day_obs_end is None else int(day_obs_end)
day_obs_date = datetime.strptime(str(day_obs), "%Y%m%d").date()
day_obs_end_date = datetime.strptime(str(day_obs_end), "%Y%m%d").date()
n_nights_range = (day_obs_end_date - day_obs_date).days + 1
if n_nights_range < 1:
    raise ValueError(f"day_obs_end ({day_obs_end}) is before day_obs ({day_obs})")
DAY_OBS_LIST = [
    int((day_obs_date + timedelta(days=i)).strftime("%Y%m%d"))
    for i in range(n_nights_range)
]
multi_night = n_nights_range > 1
range_label = f"dayObs {day_obs}" + (f" to {day_obs_end}" if multi_night else "")
verbose_output = bool(verbose_output)


def vprint(*args, **kwargs):
    """print(), only when verbose_output is True."""
    if verbose_output:
        print(*args, **kwargs)


def vdisplay(*objs):
    """display(), only when verbose_output is True."""
    if verbose_output:
        display(*objs)


print(
    f"{range_label}: {n_nights_range} night{'s' if multi_night else ''} "
    f"({'multi-night' if multi_night else 'single-night'} run); verbose_output={verbose_output}"
)

## Data sources

| Input | Source | Used for |
|---|---|---|
| **Visits** | ConsDB + FBS targets via `rubin_nights.targets_and_visits` (same query as `FBS_Targets`) | Identifying and classifying survey visits. `obs_start` / `obs_end` are used only to match each image to its shutter events. |
| **EFD events** | `lsst.sal.*` topics via `rubin_nights.connections.get_clients()` (UTC) | All times, including the gap endpoints |
| **Observatory state** | `rubin_nights.observatory_status.get_observatory_state_times` (as in ROSP-09) | Selection (step 4) |

Ranges are queried in chunks of `QUERY_CHUNK_NIGHTS` nights, with `QUERY_WORKERS` chunks running in parallel.

The [Computation](#computation) table lists the EFD topic behind each segment boundary. These topics are used only for context:

| Use | EFD topic |
|---|---|
| Filter-change exchange time | `MTCamera.logevent_startSetFilter` / `logevent_endSetFilter` |
| Deep-dive context | `ScriptQueue.logevent_script`, `MTAOS.logevent_closedLoopState`, `MTPtg.command_poriginOffset`, `MTHexapod.logevent_inPosition`, `MTDome.logevent_azMotion`, `MTCamera.logevent_ccsCommandState`, `endReadout`, `startIntegration` |

In [ ]:
# Imports, and the connection to the EFD and ConsDB.
import os
import logging
import time as _time
import warnings
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.time import Time, TimeDelta

from rubin_nights import connections
import rubin_nights.dayobs_utils as rn_dayobs
from rubin_nights.targets_and_visits import targets_and_visits

# The Sun-position code needs rubin_sim data; fall back to the shared copy at USDF.
if os.getenv("RUBIN_SIM_DATA_DIR") is None:
    for _data_dir in (
        "/sdf/data/rubin/shared/rubin_sim_data",
        "/sdf/group/rubin/shared/rubin_sim_data",
    ):
        if os.path.isdir(_data_dir):
            os.environ["RUBIN_SIM_DATA_DIR"] = _data_dir
            break

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("rubin_nights").setLevel(logging.WARNING)

# get_clients() occasionally times out; retry a few times.
for _attempt in range(4):
    try:
        endpoints = connections.get_clients()
        break
    except Exception as e:
        if _attempt == 3:
            raise
        print(f"get_clients() failed ({type(e).__name__}: {e}); retrying in 5 s")
        _time.sleep(5)
efd = endpoints["efd"]

In [ ]:
# Query window (12:00 UTC on the first dayObs to the end of the last), observable-night length per
# dayObs (Sun below -12 deg, for the nights-per-year extrapolation), and the chunked query helpers.
TWILIGHT_ALT = -12.0  # Sun altitude (deg) bounding the observable night
QUERY_CHUNK_NIGHTS = 14  # nights per ConsDB / EFD query
QUERY_WORKERS = 4  # chunks fetched in parallel
QUERY_RETRIES = 3  # attempts per chunk before failing

t_start = rn_dayobs.day_obs_to_time(day_obs)
t_end = rn_dayobs.day_obs_to_time(day_obs_end) + TimeDelta(1, format="jd")
QUERY_CHUNKS = [
    (
        t_start + TimeDelta(i, format="jd"),
        min(t_start + TimeDelta(i + QUERY_CHUNK_NIGHTS, format="jd"), t_end),
    )
    for i in range(0, n_nights_range, QUERY_CHUNK_NIGHTS)
]

# Observable-night length of every dayObs in the range (hours).
night_hours_by_night = pd.Series(
    {
        d: (lambda ss_sr: (ss_sr[1] - ss_sr[0]).jd * 24)(
            rn_dayobs.day_obs_sunset_sunrise(d, TWILIGHT_ALT)
        )
        for d in DAY_OBS_LIST
    },
    name="observable_hours",
)
night_hours = night_hours_by_night.iloc[0]  # single-night runs


def _retry(fetch, t0, t1):
    for attempt in range(QUERY_RETRIES):
        try:
            return fetch(t0, t1)
        except Exception as e:
            if attempt == QUERY_RETRIES - 1:
                raise
            print(
                f"query {t0.iso[:10]}..{t1.iso[:10]} failed ({type(e).__name__}: {e}); retrying"
            )
            _time.sleep(3)


def query_in_chunks(fetch):
    """fetch(t0, t1) over every query chunk (in parallel); the non-empty results concatenated in time order."""
    with ThreadPoolExecutor(max_workers=min(QUERY_WORKERS, len(QUERY_CHUNKS))) as pool:
        parts = list(pool.map(lambda c: _retry(fetch, *c), QUERY_CHUNKS))
    parts = [p for p in parts if p is not None and len(p) > 0]
    return pd.concat(parts) if parts else pd.DataFrame()


def efd_frame(topic, fields, query_filter=None, index=None):
    """One EFD topic over the whole window (time-indexed, UTC), filtered chunk by chunk."""

    def fetch(t0, t1):
        d = efd.select_time_series(
            topic, fields if fields else "*", t0, t1, index=index
        )
        if len(d) > 0 and query_filter is not None:
            d = d.query(query_filter)
        return d

    d = query_in_chunks(fetch)
    if len(d) > 0:
        d = d[
            ~d.index.duplicated()
        ].sort_index()  # an event on a chunk boundary comes back twice
    return d


def efd_times(topic, query_filter=None, fields=None, index=None):
    """Sorted UTC timestamps of one EFD topic over the whole window (optionally filtered)."""
    d = efd_frame(topic, fields, query_filter, index)
    return d.index if len(d) else pd.DatetimeIndex([], tz="UTC")


vprint(
    f"Query window (UTC): {t_start.iso} to {t_end.iso}; {len(QUERY_CHUNKS)} chunk(s) of "
    f"up to {QUERY_CHUNK_NIGHTS} nights"
)
if not multi_night:
    sunset12, sunrise12 = rn_dayobs.day_obs_sunset_sunrise(day_obs, TWILIGHT_ALT)
    vprint(
        f"Observable night (Sun {TWILIGHT_ALT:g} deg): {sunset12.utc.iso} to {sunrise12.utc.iso} ({night_hours:.2f} h)"
    )

## Selection

Every gap carries a `gap_class` label, so any class can be split out later.

1. **Survey visits only.** Uses the same `science_program` blocks and `observation_reason` exclusions as `FBS_Targets`.
2. **Strata.** `main_survey`, `deep_drilling` (`ddf*`) and `twilight` (`twilight*`). All three are in the sample (`SAMPLE_STRATA`).
3. **Gap classes**, based on the previous image of the same night:
   - `after_non_survey`: the previous image is not a survey visit. Set aside.
   - `filter_change`: the band changed. Set aside and reported separately.
   - `strategy_change`: the stratum changed. Kept in the sample and flagged.
   - `clean`: kept in the sample.

   Each gap is timed from the EFD shutter events. Gaps whose shutter events can't be matched are labelled `no_shutter_event` and removed. There is no cut on gap length.
4. **Observatory-state guard.** A `FAULT`, `IDLE` or `DOWNTIME` period anywhere from shutter open of visit N to shutter close of visit N+1 makes the gap `non_operating`. That gap and the gaps on either side of it are removed.
5. **Night coverage (ranges only).** A night counts only if it has at least `MIN_SAMPLE_GAPS_PER_NIGHT` gaps in the sample.

**Analysis sample:** classes `clean` and `strategy_change`, in the strata `SAMPLE_STRATA`, on the nights `NIGHTS_IN_SAMPLE`. Removed gaps appear only in the removal log.

In [ ]:
# Fetch every image in the range (ConsDB + FBS targets, as in FBS_Targets), keeping only the columns used.
VISIT_COLUMNS = [
    "visit_id", "day_obs", "seq_num", "band", "physical_filter",
    "obs_start_mjd", "obs_end_mjd", "visit_gap",
    "science_program", "observation_reason", "img_type", "target_name",
    "s_ra", "s_dec", "sky_rotation", "azimuth", "altitude",
]  # fmt: skip


def fetch_visits(t0, t1):
    v = targets_and_visits(t0, t1, endpoints, queue_index=1)[-1]
    return v[[c for c in VISIT_COLUMNS if c in v.columns]] if len(v) else v


_t0 = _time.time()
visits = query_in_chunks(fetch_visits)
if len(visits) == 0:
    raise RuntimeError(f"No visits in the ConsDB for {range_label}")
visits["day_obs"] = visits.day_obs.astype(int)
visits = (
    visits[
        visits.day_obs.isin(DAY_OBS_LIST)
    ]  # a chunk window can catch an image of the night before
    .drop_duplicates("visit_id")
    .sort_values("obs_start_mjd")
    .reset_index(drop=True)
)
vprint(
    f"{len(visits)} images over {visits.day_obs.nunique()} nights of {range_label} "
    f"(fetched in {_time.time() - _t0:.1f} s)"
)
vdisplay(
    visits.groupby(["science_program", "img_type"]).size().rename("n_images").to_frame()
)

In [ ]:
# STEP 1 + 2: survey visits only, split into strata.
# The science programs and exclusions follow the FBS_Targets notebook (LSSTCam era).
SCIENCE_PROGRAMS = [
    "BLOCK-365",
    "BLOCK-407",
    "BLOCK-408",
    "BLOCK-416",
    "BLOCK-417",
    "BLOCK-419",
    "BLOCK-421",
]

_reason = visits.observation_reason.fillna("")
is_survey = (
    visits.science_program.isin(SCIENCE_PROGRAMS)
    & (_reason != "field_survey_science")
    & (_reason != "block-t538")
    & ~_reason.str.contains("close_loop")
    & (visits.img_type == "science")
)

# Cadence stratum of a survey visit, from its observation_reason ("" for non-survey images).
visits["stratum"] = np.where(
    is_survey,
    np.select(
        [_reason.str.startswith("ddf"), _reason.str.startswith("twilight")],
        ["deep_drilling", "twilight"],
        default="main_survey",
    ),
    "",
)

vprint(f"{is_survey.sum()} survey visits of {len(visits)} images")
vdisplay(
    visits[is_survey]
    .groupby(["stratum", "observation_reason"])
    .agg(n=("visit_id", "size"), median_gap=("visit_gap", "median"))
    .round(2)
)

In [ ]:
# STEP 3: classify the gap in front of each survey visit and time it from the EFD shutter events.
SAMPLE_STRATA = [
    "main_survey",
    "deep_drilling",
    "twilight",
]  # survey tiers in the analysis sample
SAMPLE_CLASSES = ["clean", "strategy_change"]  # gap classes in the analysis sample
SHUTTER_MATCH_TOL_S = (
    1.0  # max |EFD shutter event - ConsDB time| when matching events to images
)
_NAT_NS = np.iinfo(np.int64).min  # NaT as int64 nanoseconds


def nearest_event(event_times, targets, tol_s):
    """Nearest event to each target time (both UTC), NaT when none is within tol_s (datetime64[ns], UTC)."""
    ev = pd.DatetimeIndex(event_times).asi8
    tg = pd.DatetimeIndex(targets).asi8
    if len(ev) == 0:
        return np.full(len(tg), np.datetime64("NaT"), dtype="datetime64[ns]")
    idx = np.clip(np.searchsorted(ev, tg), 1, max(len(ev) - 1, 1))
    left, right = ev[idx - 1], ev[np.minimum(idx, len(ev) - 1)]
    best = np.where(np.abs(tg - left) <= np.abs(right - tg), left, right)
    ok = (tg != _NAT_NS) & (np.abs(best - tg) <= tol_s * 1e9)
    return np.where(ok, best, _NAT_NS).view("datetime64[ns]")


def tai_mjd_to_utc(mjd):
    """TAI MJDs -> tz-aware UTC DatetimeIndex (NaN -> NaT)."""
    mjd = np.asarray(mjd, dtype=float)
    out = np.full(len(mjd), np.datetime64("NaT"), dtype="datetime64[ns]")
    ok = np.isfinite(mjd)
    if ok.any():
        out[ok] = Time(mjd[ok], format="mjd", scale="tai").utc.datetime64
    return pd.to_datetime(out, utc=True)


# Gap endpoints come from the EFD; ConsDB times only identify which shutter event belongs to which image.
ev_shutter_close = efd_times(
    "lsst.sal.MTCamera.logevent_endShutterClose", fields=["private_origin"]
)
ev_shutter_open = efd_times(
    "lsst.sal.MTCamera.logevent_startShutterOpen", fields=["private_origin"]
)
visits["t_shutter_open"] = nearest_event(
    ev_shutter_open, tai_mjd_to_utc(visits.obs_start_mjd), SHUTTER_MATCH_TOL_S
)
visits["t_shutter_close"] = nearest_event(
    ev_shutter_close, tai_mjd_to_utc(visits.obs_end_mjd), SHUTTER_MATCH_TOL_S
)

# Previous image over ALL images of the same night: a non-survey image breaks the pair, and no
# gap spans two nights.
prev = visits.groupby("day_obs")[
    [
        "visit_id",
        "stratum",
        "band",
        "obs_start_mjd",
        "t_shutter_open",
        "t_shutter_close",
    ]
].shift(1)
gaps = visits[is_survey].copy()
gaps["prev_visit_id"] = prev.loc[gaps.index, "visit_id"]
gaps["prev_stratum"] = prev.loc[gaps.index, "stratum"].fillna("")
gaps["prev_band"] = prev.loc[gaps.index, "band"]
gaps["prev_obs_start_mjd"] = prev.loc[gaps.index, "obs_start_mjd"]

gaps["gap_class"] = np.select(
    [
        gaps.prev_visit_id.isna()
        | gaps.prev_stratum.eq(""),  # previous image not a survey visit
        gaps.prev_band.ne(gaps.band),  # filter change (takes precedence)
        gaps.prev_stratum.ne(gaps.stratum),  # change of stratum
    ],
    ["after_non_survey", "filter_change", "strategy_change"],
    default="clean",
)
gaps["strategy_change"] = gaps.prev_stratum.ne(gaps.stratum) & gaps.prev_stratum.ne("")

gaps["t_close"] = pd.to_datetime(
    prev.loc[gaps.index, "t_shutter_close"], utc=True
)  # previous image
gaps["t_open"] = pd.to_datetime(gaps.t_shutter_open, utc=True)
gaps["t_prev_open"] = pd.to_datetime(prev.loc[gaps.index, "t_shutter_open"], utc=True)
gaps["t_this_close"] = pd.to_datetime(gaps.t_shutter_close, utc=True)
gaps["visit_gap_consdb"] = gaps.visit_gap  # kept for the cross-check
gaps["visit_gap"] = (
    gaps.t_open - gaps.t_close
).dt.total_seconds()  # the gap, from the EFD

no_shutter = gaps.t_close.isna() | gaps.t_open.isna()
gaps.loc[no_shutter, "gap_class"] = "no_shutter_event"

d_gap = (gaps.visit_gap - gaps.visit_gap_consdb).dropna()
vprint(
    f"Shutter events: {len(ev_shutter_close)} endShutterClose, {len(ev_shutter_open)} startShutterOpen; "
    f"{int(no_shutter.sum())} survey gaps without a matched pair (class no_shutter_event)"
)
if len(d_gap):
    vprint(
        f"EFD gap - ConsDB gap: median {d_gap.median():+.3f} s, max |diff| {d_gap.abs().max():.3f} s"
    )
vprint(
    f"{len(gaps)} survey-visit gaps; class counts: {gaps.gap_class.value_counts().to_dict()}"
)

In [ ]:
# STEP 4: observatory-state guard. A FAULT, IDLE or DOWNTIME period anywhere from shutter open (N)
# to shutter close (N+1) makes the gap non_operating; the neighbouring gaps are removed too.
# exclude_cause records which state was hit, for the removal log.
from rubin_nights import observatory_status

EXCLUDE_STATES = ["FAULT", "IDLE", "DOWNTIME"]
EXCLUDE_NEIGHBOR_GAPS = True  # also remove the gaps on either side of an affected one
EXCLUDED_CLASSES = ["non_operating", "no_shutter_event"]  # out of the analysis

# rubin_nights raises a harmless pandas FutureWarning here; silence just that one.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", category=FutureWarning, module="rubin_nights.observatory_status"
    )
    obs_states = query_in_chunks(
        lambda t0, t1: observatory_status.get_observatory_state_times(t0, t1, efd)
    ).reset_index(drop=True)
vprint(
    f"Observatory states over {range_label}: "
    f"{obs_states.groupby('type').hours.sum().round(2).to_dict() if len(obs_states) else 'none recorded'} (hours)"
)


def _ns(x):
    """Datetimes (naive = UTC, or tz-aware) as int64 nanoseconds since epoch, UTC; NaT -> int64 min."""
    idx = pd.DatetimeIndex(x)
    return (idx.tz_localize("UTC") if idx.tz is None else idx.tz_convert("UTC")).asi8


def overlaps_states(states, w0, w1):
    """True where the window [w0, w1] overlaps any period of the given state types."""
    w0n, w1n = _ns(w0), _ns(w1)
    hit = np.zeros(len(w0n), dtype=bool)
    periods = (
        obs_states[obs_states.type.isin(states)] if len(obs_states) else obs_states
    )
    if len(periods) == 0:
        return hit
    p_start, p_end = _ns(periods.start), _ns(periods.end)
    order = np.argsort(p_start)
    p_start, p_end_max = p_start[order], np.maximum.accumulate(p_end[order])
    i = np.searchsorted(p_start, w1n, side="left") - 1  # last period starting before w1
    ok = (w0n != _NAT_NS) & (w1n != _NAT_NS) & (i >= 0)
    hit[ok] = p_end_max[i[ok]] > w0n[ok]
    return hit


# Guard window: shutter open (N) to shutter close (N+1); ConsDB times only as a fallback.
w0 = gaps.t_prev_open.fillna(
    pd.Series(tai_mjd_to_utc(gaps.prev_obs_start_mjd), index=gaps.index)
)
w1 = gaps.t_this_close.fillna(
    pd.Series(tai_mjd_to_utc(gaps.obs_end_mjd), index=gaps.index)
)

_cause = pd.Series("", index=gaps.index)
for _state in EXCLUDE_STATES:
    _hit = pd.Series(overlaps_states([_state], w0, w1), index=gaps.index)
    _cause = _cause.mask(_hit, (_cause + "+" + _state).str.lstrip("+"))
gaps["state_overlap"] = _cause.ne("")
_so = gaps.groupby("day_obs").state_overlap
gaps["state_neighbor"] = (
    _so.shift(1).eq(True) | _so.shift(-1).eq(True)
) & ~gaps.state_overlap
gaps["exclude_cause"] = _cause
if EXCLUDE_NEIGHBOR_GAPS:
    gaps.loc[gaps.state_neighbor, "exclude_cause"] = "neighbour of an affected gap"

affected = gaps.state_overlap | (
    gaps.state_neighbor if EXCLUDE_NEIGHBOR_GAPS else False
)
gaps.loc[affected, "gap_class"] = "non_operating"  # overrides every other class

vprint(
    f"Gaps overlapping a {EXCLUDE_STATES} period: {int(gaps.state_overlap.sum())} "
    f"({gaps.loc[gaps.state_overlap, 'exclude_cause'].value_counts().to_dict()}); "
    f"neighbouring gaps: {int(gaps.state_neighbor.sum())} (excluded: {EXCLUDE_NEIGHBOR_GAPS})"
)

summary = (
    gaps.groupby(["stratum", "gap_class"])
    .agg(
        n=("visit_id", "size"),
        median_gap=("visit_gap", "median"),
        total_min=("visit_gap", lambda s: s.sum() / 60),
    )
    .round(2)
)
vdisplay(summary)

In [ ]:
# STEP 5: night coverage. In a range, a night needs MIN_SAMPLE_GAPS_PER_NIGHT gaps in the sample;
# near-empty nights would add a full night to the nights-per-year denominator.
MIN_SAMPLE_GAPS_PER_NIGHT = 100


def day_obs_of(times):
    """dayObs (int YYYYMMDD) of UTC timestamps: the date at UTC-12h."""
    t = pd.to_datetime(_ns(times) - 12 * 3600 * 10**9, utc=True)
    return pd.Index(t.strftime("%Y%m%d").astype(int))


_in_class = gaps.stratum.isin(SAMPLE_STRATA) & gaps.gap_class.isin(SAMPLE_CLASSES)
night_table = pd.DataFrame(index=pd.Index(DAY_OBS_LIST, name="day_obs"))
night_table["observable_hours"] = night_hours_by_night.round(2)
night_table["images"] = visits.groupby("day_obs").size()
night_table["survey_visits"] = gaps.groupby("day_obs").size()
night_table["efd_shutter_closes"] = day_obs_of(ev_shutter_close).value_counts()
night_table["excluded_gaps"] = (
    gaps[gaps.gap_class.isin(EXCLUDED_CLASSES)].groupby("day_obs").size()
)
night_table["sample_gaps"] = gaps[_in_class].groupby("day_obs").size()
_count_cols = night_table.columns.drop("observable_hours")
night_table[_count_cols] = night_table[_count_cols].fillna(0).astype(int)
_min_gaps = MIN_SAMPLE_GAPS_PER_NIGHT if multi_night else 1
night_table["status"] = np.select(
    [
        night_table.images == 0,
        night_table.survey_visits == 0,
        night_table.efd_shutter_closes == 0,
        night_table.sample_gaps < _min_gaps,
    ],
    [
        "no images in ConsDB",
        "no survey visits",
        "no EFD shutter events",
        f"fewer than {_min_gaps} sample gaps",
    ],
    default="in sample",
)
NIGHTS_IN_SAMPLE = night_table.index[night_table.status == "in sample"].tolist()
N_NIGHTS = len(NIGHTS_IN_SAMPLE)
if N_NIGHTS == 0:
    raise RuntimeError(f"No night of {range_label} has gaps in the analysis sample")


def analysis_sample():
    """The gaps that are decomposed and reported: SAMPLE_STRATA x SAMPLE_CLASSES, nights in NIGHTS_IN_SAMPLE."""
    return gaps[
        gaps.stratum.isin(SAMPLE_STRATA)
        & gaps.gap_class.isin(SAMPLE_CLASSES)
        & gaps.day_obs.isin(NIGHTS_IN_SAMPLE)
    ]


sample = analysis_sample()
if multi_night:
    print(f"Night coverage, {range_label} ({n_nights_range} nights):")
    for status, n in night_table.status.value_counts().items():
        print(f"  {n:4d} nights  {status}")
    _dropped = night_table[
        (night_table.status != "in sample") & (night_table.sample_gaps > 0)
    ]
    if len(_dropped):
        print(
            f"  (dropped for too few gaps: {', '.join(f'{d} ({n})' for d, n in _dropped.sample_gaps.items())})"
        )
    vdisplay(
        night_table[(night_table.images > 0) | (night_table.efd_shutter_closes > 0)]
    )
print(
    f"Analysis sample: {len(sample)} gaps over {N_NIGHTS} night{'s' if N_NIGHTS > 1 else ''} "
    f"(by stratum: {sample.stratum.value_counts().to_dict()}; "
    f"strategy changes: {int(sample.strategy_change.sum())})"
)

<a id="computation"></a>
## Computation

The gap is cut at **markers**: EFD events found between `t_close` and `t_open`. Each segment starts where the previous one ends, beginning at `endShutterClose` of visit N. Markers are either SAL commands or log events.

In each row, line 1 is the event that ends the segment and line 2 is what happens during it.

| # | Segment · owner | Ends at (marker) → what fills it | Code |
|---|---|---|---|
| 1 | **Script startup**<br>ScriptQueue | first main-queue `Script.state = RUNNING` (`t_run`)<br>script N finishes; the queue starts script N+1 | [Script states](https://github.com/lsst-ts/ts_xml/blob/main/python/lsst/ts/xml/enums/Script.py) (ts_xml `ScriptState`; set by ts_salobj `BaseScript`) |
| 2 | **Script & TCS pre-slew checks**<br>Scripts / MTCS | first `MTM1M3.setSlewFlag` (`t_flag_set`)<br>feasibility checks; MTCS cable-wrap, trackId and hexapod-compensation checks | [`BaseTrackTargetAndTakeImage.run`](https://github.com/lsst-ts/ts_standardscripts/blob/main/python/lsst/ts/standardscripts/base_track_target_and_take_image.py) → [mtcs.py `_slew_to`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/maintel/mtcs.py) |
| 3 | **Telescope prep (M1M3)**<br>M1M3 | pre-slew `MTAOS.issueCorrection` (`t_aos_pre`), else the slew command<br>booster valves open, hardpoint corrections | [mtcs.py `m1m3_booster_valve`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/maintel/mtcs.py) |
| 4 | **AOS Correction (pre-slew)**<br>AOS | `MTPtg.raDecTarget` (`t_slew_cmd`)<br>a waiting correction is applied before the slew · 0 if none | [mtcs.py `handle_aos_close_loop`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/maintel/mtcs.py) |
| 5 | **Slew start latency**<br>TMA / pointing | first `MTMount` azimuth/elevation `inPosition → False` (`t_motion`)<br>slew command until the first axis moves | [base_tcs.py `slew`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/base_tcs.py) |
| 6 | **Mount slew+settle**<br>TMA | last az/el `inPosition → True` (`t_mount_done`)<br>axis motion and settling · sub-category: limiting axis (azimuth or elevation) | [mtcs.py `wait_for_mtmount_inposition`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/maintel/mtcs.py) |
| 7 | **Rotator catch-up**<br>Rotator | last `MTRotator.inPosition → True` (`t_rot_done`)<br>rotator time beyond the mount · usually 0 | [mtcs.py `wait_for_rotator_inposition`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/maintel/mtcs.py) |
| 8 | **Dome catch-up**<br>Dome | last `MTDomeTrajectory.telescopeVignetted → NO` (`t_dome_ok`)<br>dome time beyond mount + rotator · usually 0 | [mtcs.py `wait_for_dome_inposition`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/maintel/mtcs.py) |
| 9 | **In-position checks**<br>OCS / MTCS | second `setSlewFlag` (`t_post_flag`), else `takeImages`<br>MTCS confirmations, including timeouts that leave no events | [mtcs.py `_slew_to` / `wait_for_inposition`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/maintel/mtcs.py) |
| 10 | **AOS Correction (post-slew)**<br>AOS | `MTAOS.closedLoopState → WAITING_IMAGE` (`t_aos_applied`)<br>correction that became ready after the slew started · 0 if none | [script `wait_mtaos_idle`](https://github.com/lsst-ts/ts_maintel_standardscripts/blob/main/python/lsst/ts/maintel/standardscripts/track_target_and_take_image_lsstcam.py) · [mtaos.py `handle_corrections`](https://github.com/lsst-ts/ts_mtaos/blob/main/python/lsst/ts/mtaos/mtaos.py) |
| 11 | **AOS pointing correction**<br>AOS / TMA | second `MTM1M3.clearSlewFlag` (`t_post_clear`)<br>settling onto the `MTPtg.poriginOffset` sent with the correction · 0 if none | [mtaos.py `issue_pointing_correction`](https://github.com/lsst-ts/ts_mtaos/blob/main/python/lsst/ts/mtaos/mtaos.py) |
| 12 | **Camera dispatch**<br>OCS / Camera | `MTCamera.takeImages` (`t_takeimages`)<br>MTAOS-idle and CCS-idle checks · only after a post-slew correction, else 0 | [script `_take_data`](https://github.com/lsst-ts/ts_maintel_standardscripts/blob/main/python/lsst/ts/maintel/standardscripts/track_target_and_take_image_lsstcam.py) · [base_camera.py `expose`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/base_camera.py) |
| 13 | **Camera start**<br>Camera / CCS | `MTCamera.startShutterOpen` (`t_open`)<br>camera executing `takeImages` (~0.15 s) | [MTCamera events](https://github.com/lsst-ts/ts_xml/blob/main/python/lsst/ts/xml/data/sal_interfaces/MTCamera/MTCamera_Events.xml) |

Rules:

- **Sum to the gap.** Markers are clipped into the gap. A missing marker takes the neighbouring boundary, so its segment is 0.
- **Marginal attribution.** The rotator and dome are charged only for the time they take beyond the mount.
- **Mount sub-category.** All of segment 6 is charged to the axis that reaches position last (`slew_limited_by`, `c_slew_az` / `c_slew_el`).
- **Post-slew boundary.** Markers for segments 6–8 are searched only before `t_post_flag`, so any re-pointing after a post-slew correction falls in segments 10–11.

Caveats:

- *Camera dispatch* can only be separated after a post-slew correction. Otherwise it stays inside *In-position checks* (~0.1–0.5 s).
- *In-position checks* and *AOS pointing correction* are mostly in-position timeouts that leave no events.
- If the mount doesn't move, segments 5–6 are 0 and the time goes to the slower system, usually *Rotator catch-up*.

In [ ]:
# EFD marker topics for the whole range (UTC timestamps).
_t0 = _time.time()
ev_flag_set = efd_times(
    "lsst.sal.MTM1M3.command_setSlewFlag", fields=["private_identity"]
)
ev_flag_clear = efd_times(
    "lsst.sal.MTM1M3.command_clearSlewFlag", fields=["private_identity"]
)
ev_aos = efd_times(
    "lsst.sal.MTAOS.command_issueCorrection", fields=["private_sndStamp"]
)
# MTAOS closedLoopState WAITING_IMAGE (= 2): MTAOS has finished applying a correction.
ev_aos_applied = efd_times(
    "lsst.sal.MTAOS.logevent_closedLoopState", "state == 2", ["state"]
)
ev_slew_cmd = efd_times("lsst.sal.MTPtg.command_raDecTarget", fields=["targetName"])
_az = efd_frame("lsst.sal.MTMount.logevent_azimuthInPosition", ["inPosition"])
_el = efd_frame("lsst.sal.MTMount.logevent_elevationInPosition", ["inPosition"])


def _in_position(frame, value):
    """Timestamps where inPosition == value (empty when the topic returned nothing)."""
    if len(frame) == 0:
        return pd.DatetimeIndex([], tz="UTC")
    return frame.index[frame.inPosition == value]


ev_az_done, ev_el_done = _in_position(_az, True), _in_position(_el, True)
ev_az_move = _in_position(_az, False)  # axis leaves position: motion starts
ev_el_move = _in_position(_el, False)
ev_rot_done = efd_times(
    "lsst.sal.MTRotator.logevent_inPosition", "inPosition == True", ["inPosition"]
)
ev_dome_ok = efd_times(
    "lsst.sal.MTDomeTrajectory.logevent_telescopeVignetted",
    "vignetted == 1",
    ["vignetted"],
)
ev_takeimages = efd_times("lsst.sal.MTCamera.command_takeImages", fields=["numImages"])
ev_filter_start = efd_times(
    "lsst.sal.MTCamera.logevent_startSetFilter", fields=["filterName"]
)
ev_filter_end = efd_times(
    "lsst.sal.MTCamera.logevent_endSetFilter", fields=["filterName"]
)

# Main-queue scripts (salIndex 1xxxxx) entering RUNNING (ScriptState.RUNNING == 3).
ev_script_run = efd_times(
    "lsst.sal.Script.logevent_state",
    "salIndex < 200000 and state == 3",
    ["state", "salIndex"],
)

for name, ev in [
    ("setSlewFlag", ev_flag_set),
    ("clearSlewFlag", ev_flag_clear),
    ("issueCorrection", ev_aos),
    ("AOS WAITING_IMAGE", ev_aos_applied),
    ("raDecTarget", ev_slew_cmd),
    ("azInPosition=False", ev_az_move),
    ("elInPosition=False", ev_el_move),
    ("azInPosition=True", ev_az_done),
    ("elInPosition=True", ev_el_done),
    ("rotInPosition=True", ev_rot_done),
    ("domeVignetted=NO", ev_dome_ok),
    ("takeImages", ev_takeimages),
    ("script RUNNING", ev_script_run),
    ("startSetFilter", ev_filter_start),
    ("endSetFilter", ev_filter_end),
]:
    vprint(f"{name:20s} {len(ev):7d} events")
vprint(f"(EFD marker topics fetched in {_time.time() - _t0:.1f} s)")

In [ ]:
# Assign markers to each gap: the FIRST event in the window for commands and script states, the
# LAST for in-position / vignetting transitions and takeImages.
def _nat_series(index):
    return pd.Series(pd.NaT, index=index, dtype="datetime64[ns, UTC]")


def first_in_window(event_times, lo, hi):
    """First event in [lo, hi] for each window; NaT when there is none."""
    ev = pd.DatetimeIndex(event_times).asi8
    if len(ev) == 0:
        return _nat_series(lo.index)
    lo_ns, hi_ns = pd.DatetimeIndex(lo).asi8, pd.DatetimeIndex(hi).asi8
    idx = np.searchsorted(ev, lo_ns, side="left")
    cand = np.where(
        idx < len(ev), ev[np.minimum(idx, len(ev) - 1)], np.iinfo(np.int64).max
    )
    found = (idx < len(ev)) & (cand <= hi_ns)
    out = np.where(found, cand, pd.NaT.value)
    return pd.Series(pd.to_datetime(out, utc=True), index=lo.index).mask(~found)


def last_in_window(event_times, lo, hi):
    """Last event in [lo, hi] for each window; NaT when there is none."""
    ev = pd.DatetimeIndex(event_times).asi8
    if len(ev) == 0:
        return _nat_series(lo.index)
    lo_ns, hi_ns = pd.DatetimeIndex(lo).asi8, pd.DatetimeIndex(hi).asi8
    idx = np.searchsorted(ev, hi_ns, side="right") - 1
    cand = np.where(idx >= 0, ev[np.maximum(idx, 0)], np.iinfo(np.int64).min)
    found = (idx >= 0) & (cand >= lo_ns)
    out = np.where(found, cand, pd.NaT.value)
    return pd.Series(pd.to_datetime(out, utc=True), index=lo.index).mask(~found)


g_ok = gaps.t_close.notna() & gaps.t_open.notna()
lo, hi = gaps.loc[g_ok, "t_close"], gaps.loc[g_ok, "t_open"]
ONE_NS = pd.Timedelta(1, "ns")
m = pd.DataFrame(index=lo.index)
m["t_run"] = first_in_window(ev_script_run, lo, hi)  # script N+1 goes RUNNING
m["t_flag_set"] = first_in_window(ev_flag_set, lo, hi)
m["t_slew_cmd"] = first_in_window(ev_slew_cmd, lo, hi)
slew_or_open = m.t_slew_cmd.fillna(hi)
# Pre-slew AOS: issueCorrection before the slew command.
m["t_aos_pre"] = first_in_window(ev_aos, lo, slew_or_open)
# Post-slew AOS: a second setSlewFlag after the slew command, followed by issueCorrection ->
# WAITING_IMAGE -> clearSlewFlag, then takeImages. Accepted only if issueCorrection follows.
second_flag = first_in_window(ev_flag_set, slew_or_open + ONE_NS, hi)
m["t_aos_post"] = first_in_window(ev_aos, second_flag.fillna(hi), hi)
m["t_post_flag"] = second_flag.where(m.t_aos_post.notna())
post_or_open = m.t_post_flag.fillna(hi)
m["t_aos_applied"] = first_in_window(
    ev_aos_applied, m.t_aos_post.fillna(hi), hi
)  # WAITING_IMAGE
m["t_post_clear"] = last_in_window(ev_flag_clear, post_or_open, hi).where(
    m.t_post_flag.notna()
)

# Slew markers stop at the post-slew AOS cycle, so its own in-position toggles are charged to it.
m["t_az_move"] = first_in_window(ev_az_move, slew_or_open, post_or_open)
m["t_el_move"] = first_in_window(ev_el_move, slew_or_open, post_or_open)
m["t_motion"] = m[["t_az_move", "t_el_move"]].min(axis=1)  # first axis leaves position
m["t_az_done"] = last_in_window(ev_az_done, lo, post_or_open)
m["t_el_done"] = last_in_window(ev_el_done, lo, post_or_open)
m["t_mount_done"] = m[["t_az_done", "t_el_done"]].max(axis=1)
m["t_rot_done"] = last_in_window(ev_rot_done, lo, post_or_open)
m["t_dome_ok"] = last_in_window(ev_dome_ok, lo, post_or_open)
m["t_takeimages"] = last_in_window(ev_takeimages, lo, hi)
for c in m.columns:
    gaps[c] = m[c].reindex(gaps.index)  # NaT for gaps without shutter events
gaps["second_flag_cycle"] = (second_flag.notna()).reindex(gaps.index, fill_value=False)

# Marker completeness in the analysis sample.
sample = analysis_sample()
marker_cols = [
    "t_run",
    "t_flag_set",
    "t_aos_pre",
    "t_slew_cmd",
    "t_motion",
    "t_mount_done",
    "t_rot_done",
    "t_dome_ok",
    "t_post_flag",
    "t_aos_post",
    "t_aos_applied",
    "t_post_clear",
    "t_takeimages",
]
vprint(f"Marker completeness over {len(sample)} gaps in the analysis sample:")
for c in marker_cols:
    vprint(f"  {c:15s} {sample[c].notna().mean():6.1%}")
post = sample[sample.t_post_flag.notna()]
vprint(
    f"Post-slew AOS cycles: {len(post)}; with WAITING_IMAGE found: {post.t_aos_applied.notna().sum()}, "
    f"with the closing clearSlewFlag found: {post.t_post_clear.notna().sum()}"
)
# Should be 0: a second slew-flag cycle without issueCorrection would land in In-position checks.
n_flag_no_aos = int((sample.second_flag_cycle & sample.t_aos_post.isna()).sum())
vprint(
    f"Second slew-flag cycles WITHOUT an issueCorrection in the sample: {n_flag_no_aos}"
)

In [ ]:
# THE DECOMPOSITION: cut each gap into the thirteen serial segments between consecutive markers.
# Each boundary is clipped between its neighbours and a missing marker takes the previous
# boundary, so the segments sum exactly to the gap.
def seconds(a, b):
    """(b - a) in seconds, elementwise, NaN-safe."""
    return (b - a).dt.total_seconds()


d = gaps.loc[gaps.t_close.notna() & gaps.t_open.notna()].copy()

# Boundary chain: t_close <= b_run <= ... <= b_take <= t_open.
b_run = d.t_run.fillna(d.t_close).clip(
    lower=d.t_close, upper=d.t_open
)  # script RUNNING
b_flag = d.t_flag_set.fillna(b_run).clip(
    lower=b_run, upper=d.t_open
)  # M1M3 slew flag set
b_slew = d.t_slew_cmd.fillna(b_flag).clip(lower=b_flag, upper=d.t_open)  # raDecTarget
b_aos_pre = d.t_aos_pre.fillna(b_slew).clip(
    lower=b_flag, upper=b_slew
)  # pre-slew issueCorrection
b_motion = d.t_motion.fillna(b_slew).clip(
    lower=b_slew, upper=d.t_open
)  # first axis moves
b_mount = d.t_mount_done.fillna(b_motion).clip(
    lower=b_motion, upper=d.t_open
)  # az+el in position
b_rot = d.t_rot_done.fillna(b_mount).clip(
    lower=b_mount, upper=d.t_open
)  # rotator in position
b_dome = d.t_dome_ok.fillna(b_rot).clip(
    lower=b_rot, upper=d.t_open
)  # dome not vignetting
b_take = d.t_takeimages.fillna(d.t_open).clip(
    lower=b_dome, upper=d.t_open
)  # takeImages
b_post = d.t_post_flag.fillna(b_take).clip(
    lower=b_dome, upper=b_take
)  # 2nd setSlewFlag
b_post_clear = d.t_post_clear.fillna(b_take).clip(
    lower=b_post, upper=b_take
)  # 2nd clearSlewFlag
b_applied = d.t_aos_applied.fillna(b_post_clear).clip(
    lower=b_post, upper=b_post_clear
)  # AOS WAITING_IMAGE

d["c_script_startup"] = seconds(d.t_close, b_run)
d["c_preslew"] = seconds(b_run, b_flag)
d["c_slew_prep"] = seconds(b_flag, b_aos_pre)
d["c_aos_pre"] = seconds(b_aos_pre, b_slew)
d["c_slew_latency"] = seconds(b_slew, b_motion)
d["c_slew_mount"] = seconds(b_motion, b_mount)
d["c_settle"] = seconds(b_mount, b_rot)  # rotator catch-up
d["c_dome"] = seconds(b_rot, b_dome)
d["c_inpos"] = seconds(b_dome, b_post)
d["c_aos_post"] = seconds(b_post, b_applied)
d["c_aos_pointing"] = seconds(b_applied, b_post_clear)
d["c_dispatch"] = seconds(b_post_clear, b_take)
d["c_cam_start"] = seconds(b_take, d.t_open)

# Mount sub-category: the whole mount segment goes to the axis that reaches position last.
# Per-axis motion times (az_axis_s, el_axis_s) overlap and are diagnostics only.
az_limits = d.t_az_done.notna() & ~(d.t_el_done > d.t_az_done)
d["slew_limited_by"] = np.where(
    d.t_mount_done.isna(),
    "none (mount did not move)",
    np.where(az_limits, "azimuth", "elevation"),
)
d["c_slew_az"] = d.c_slew_mount.where(az_limits, 0.0)
d["c_slew_el"] = d.c_slew_mount - d.c_slew_az
d["az_axis_s"] = seconds(b_motion, d.t_az_done.clip(lower=b_motion, upper=b_mount))
d["el_axis_s"] = seconds(b_motion, d.t_el_done.clip(lower=b_motion, upper=b_mount))

# The thirteen segments, in time order, with display names and colours.
SEGMENTS = {
    "c_script_startup": ("Script startup", "#8c6bb1"),
    "c_preslew": ("Script & TCS pre-slew checks", "#9e9ac8"),
    "c_slew_prep": ("Telescope prep (M1M3)", "#9ecae1"),
    "c_aos_pre": ("AOS Correction (pre-slew)", "#fd8d3c"),
    "c_slew_latency": ("Slew start latency", "#c6dbef"),
    "c_slew_mount": ("Mount slew+settle", "#3182bd"),
    "c_settle": ("Rotator catch-up", "#74c476"),
    "c_dome": ("Dome catch-up", "#c7a44a"),
    "c_inpos": ("In-position checks", "#66c2a5"),
    "c_aos_post": ("AOS Correction (post-slew)", "#d94801"),
    "c_aos_pointing": ("AOS pointing correction", "#fdae6b"),
    "c_dispatch": ("Camera dispatch", "#bc80bd"),
    "c_cam_start": ("Camera start", "#e377c2"),
}
SUBCOMPONENTS = list(SEGMENTS)
MECH_COMPONENTS = [name for name, _ in SEGMENTS.values()]
SEGMENT_NAME = {col: name for col, (name, _) in SEGMENTS.items()}
SEGMENT_COLOR = {name: color for _, (name, color) in SEGMENTS.items()}
for col, name in SEGMENT_NAME.items():
    d[name] = d[col]

# A segment shorter than PRESENT_S counts as "did not occur" (boundary slivers, not mechanism
# time). Affects only min and present_in; medians, means and shares count every gap.
PRESENT_S = 0.03

# Join back onto every gap (re-runnable: earlier columns are replaced).
_new_cols = [
    c
    for c in d.columns
    if c not in gaps.columns
    or c.startswith("c_")
    or c in MECH_COMPONENTS
    or c in ("slew_limited_by", "az_axis_s", "el_axis_s")
]
gaps = gaps.drop(columns=[c for c in _new_cols if c in gaps.columns]).join(d[_new_cols])

# Validation 1: segments sum to the gap; az + el sum to the mount segment.
sample = analysis_sample()
resid = (sample[SUBCOMPONENTS].sum(axis=1) - sample.visit_gap).abs()
vprint(
    f"max |sum(segments) - gap| over {len(sample)} gaps in the analysis sample: {resid.max():.4f} s"
)
assert resid.max() < 0.05, "segments do not sum to the gap"
resid_axes = (sample.c_slew_az + sample.c_slew_el - sample.c_slew_mount).abs().max()
vprint(f"max |azimuth + elevation sub-categories - mount segment|: {resid_axes:.4f} s")

# Validation 2: AOS segments and camera dispatch are zero when no correction was issued.
no_post = sample.t_post_flag.isna()
for col, mask in [
    ("c_aos_pre", sample.t_aos_pre.isna()),
    ("c_aos_post", no_post),
    ("c_aos_pointing", no_post),
    ("c_dispatch", no_post),
]:
    leak = sample.loc[mask, col].abs().max()
    vprint(
        f"max {col} seconds charged to gaps without that correction: "
        f"{0.0 if np.isnan(leak) else leak:.4f} s"
    )

## Deep dive: example gaps, event by event

These cells check the decomposition against individual gaps. They take the gaps in the sample closest to the **10th percentile**, the **median** and the **99th percentile**, plus a typical gap with a **post-slew AOS correction** (if there is one). Each is drawn as two panels on a shared time axis:

- **Top, critical path:** one bar per segment in time order, labelled with its duration. Together the bars cover the gap from shutter close to shutter open.
- **Bottom, raw events:** one row per subsystem, with a tick for each EFD event. `F`/`T` mark `inPosition` going False/True. Labels are thinned where events cluster; with `verbose_output` the full event list is printed.

How the event rows map onto the segments:

- **Scripts / queue:** script N goes `done` and N+1 goes `run` (*Script startup*). Checkpoints appear as context only.
- **M1M3 flag:** the first `set` ends the pre-slew checks. A second `set`/`clear` pair brackets a post-slew AOS cycle: `set` → `wait img` is *AOS Correction (post-slew)*, `wait img` → `clear` is *AOS pointing correction*, and `clear` → `takeImages` is *Camera dispatch*.
- **Mount / pointing:** the slew command, then axes leaving (`F`) and reaching (`T`) position. The last `T` ends *Mount slew+settle*.
- **Rotator / hexapods / dome:** the catch-ups and the *In-position checks*.

Each figure has its own x axis, so compare durations using the bar labels (or set `COMMON_TIME_AXIS = True`). To draw the gaps on either side of one visit, set `inspect_visit_id` in the parameters cell.

In [ ]:
# Pick the example gaps (10th / 50th / 99th percentile, plus a typical post-slew AOS gap) and
# fetch every EFD event inside each.
sample = analysis_sample()

DEEP_DIVE_PICKS = {
    "short (10th percentile)": 0.10,
    "typical (median)": 0.50,
    "long (99th percentile)": 0.99,
}
deep_gaps = {}
for pick_label, q in DEEP_DIVE_PICKS.items():
    target = sample.visit_gap.quantile(q)
    deep_gaps[pick_label] = sample.iloc[
        (sample.visit_gap - target).abs().argsort().iloc[0]
    ]
with_post_aos = sample[sample.t_aos_post.notna()]
if len(with_post_aos) > 0:
    target = with_post_aos.visit_gap.median()
    deep_gaps["post-slew AOS correction (median of those gaps)"] = with_post_aos.iloc[
        (with_post_aos.visit_gap - target).abs().argsort().iloc[0]
    ]

# Topics shown in the deep dive: the segment markers plus context (script/queue states, hexapods,
# dome, CCS command state, MTAOS loop state, pointing offset).
DEEP_DIVE_TOPICS = [
    ("lsst.sal.MTCamera.logevent_endShutterClose", []),
    ("lsst.sal.MTCamera.logevent_endReadout", ["imageName"]),
    (
        "lsst.sal.ScriptQueue.logevent_script",
        ["scriptSalIndex", "processState", "scriptState"],
    ),
    ("lsst.sal.Script.logevent_state", ["state", "lastCheckpoint", "salIndex"]),
    ("lsst.sal.MTM1M3.command_setSlewFlag", ["private_identity"]),
    ("lsst.sal.MTM1M3.command_clearSlewFlag", ["private_identity"]),
    ("lsst.sal.MTAOS.command_issueCorrection", ["private_identity"]),
    ("lsst.sal.MTAOS.logevent_closedLoopState", ["state"]),
    ("lsst.sal.MTPtg.command_raDecTarget", ["targetName"]),
    ("lsst.sal.MTPtg.command_poriginOffset", ["private_identity"]),
    ("lsst.sal.MTMount.logevent_azimuthInPosition", ["inPosition"]),
    ("lsst.sal.MTMount.logevent_elevationInPosition", ["inPosition"]),
    ("lsst.sal.MTRotator.logevent_inPosition", ["inPosition"]),
    ("lsst.sal.MTHexapod.logevent_inPosition", ["inPosition", "salIndex"]),
    ("lsst.sal.MTDome.logevent_azMotion", ["inPosition"]),
    ("lsst.sal.MTDomeTrajectory.logevent_telescopeVignetted", ["vignetted"]),
    ("lsst.sal.MTCamera.logevent_ccsCommandState", ["substate"]),
    ("lsst.sal.MTCamera.command_takeImages", ["numImages"]),
    ("lsst.sal.MTCamera.logevent_startIntegration", ["imageName"]),
    ("lsst.sal.MTCamera.logevent_startShutterOpen", []),
]


def fetch_gap_events(g, pad_s=2):
    """Deep-dive EFD events in [t_close - pad, t_open + pad] of gap row `g`, time-sorted.

    Returns (timestamp, name, info-dict) with name like 'MTCamera.endShutterClose'.
    """
    t0 = Time(g.t_close.tz_localize(None)) - TimeDelta(pad_s, format="sec")
    t1 = Time(g.t_open.tz_localize(None)) + TimeDelta(pad_s, format="sec")
    events = []
    for topic, fields in DEEP_DIVE_TOPICS:
        try:
            dd = efd.select_time_series(topic, fields if fields else "*", t0, t1)
        except Exception:
            continue  # topic absent in this EFD epoch
        name = topic.removeprefix("lsst.sal.").replace("logevent_", "")
        for ts, row in dd.iterrows():
            events.append((ts, name, {f: row[f] for f in fields if f in dd.columns}))
    return sorted(events, key=lambda e: e[0])


deep_events_by_pick = {}
for pick_label, g in deep_gaps.items():
    deep_events_by_pick[pick_label] = fetch_gap_events(g)
    vprint(
        f"===== {pick_label}: gap before visit {g.visit_id} "
        f"({g.visit_gap:.2f} s, {g.observation_reason}, band {g.prev_band}->{g.band}) ====="
    )
    for ts, name, info in deep_events_by_pick[pick_label]:
        dt = (ts - g.t_close).total_seconds()
        info_str = ", ".join(f"{k}={v}" for k, v in info.items())
        vprint(f"{dt:+8.2f} s  {name:34s} {info_str}")
    vprint()

if not verbose_output:
    for pick_label, g in deep_gaps.items():
        print(
            f"{pick_label}: gap before visit {g.visit_id} ({g.visit_gap:.2f} s, {g.observation_reason})"
        )

In [ ]:
# Deep-dive figures: critical path (top) and raw EFD events (bottom) on a shared time axis.
SCRIPT_STATE = {
    0: "?",
    1: "unconf",
    2: "conf",
    3: "run",
    4: "paused",
    5: "ending",
    6: "stopping",
    7: "failing",
    8: "done",
    9: "stopped",
    10: "failed",
    11: "conf_failed",
}
PROCESS_STATE = {
    1: "loading",
    2: "configured",
    3: "running",
    4: "done",
    5: "loadfail",
    6: "conffail",
    7: "terminated",
}
AOS_LOOP_STATE = {
    1: "idle",
    2: "wait img",
    3: "processing",
    4: "WAIT APPLY",
    5: "applying",
    6: "error",
}
EVENT_LABEL = {
    "MTCamera.endShutterClose": "shutter close",
    "MTCamera.endReadout": "end readout",
    "MTCamera.command_takeImages": "takeImages",
    "MTCamera.startIntegration": "start integ",
    "MTCamera.startShutterOpen": "shutter open",
    "MTM1M3.command_setSlewFlag": "set",
    "MTM1M3.command_clearSlewFlag": "clear",
    "MTAOS.command_issueCorrection": "issueCorrection",
    "MTPtg.command_raDecTarget": "slew cmd",
    "MTPtg.command_poriginOffset": "ptg offset",
    "MTMount.azimuthInPosition": "az",
    "MTMount.elevationInPosition": "el",
    "MTRotator.inPosition": "rot",
    "MTDome.azMotion": "dome",
    "MTDomeTrajectory.telescopeVignetted": "vig",
}
# (row label, event names, colour, label_events) -- label_events=False plots ticks only.
ROW_STREAMS = [
    (
        "camera",
        [
            "MTCamera.endShutterClose",
            "MTCamera.endReadout",
            "MTCamera.ccsCommandState",
            "MTCamera.command_takeImages",
            "MTCamera.startIntegration",
            "MTCamera.startShutterOpen",
        ],
        "k",
        True,
    ),
    ("scripts / queue", ["ScriptQueue.script", "Script.state"], "#8c6bb1", True),
    (
        "M1M3 flag",
        ["MTM1M3.command_setSlewFlag", "MTM1M3.command_clearSlewFlag"],
        "#9ecae1",
        True,
    ),
    (
        "AOS",
        ["MTAOS.command_issueCorrection", "MTAOS.closedLoopState"],
        "#fd8d3c",
        True,
    ),
    (
        "mount / pointing",
        [
            "MTPtg.command_raDecTarget",
            "MTPtg.command_poriginOffset",
            "MTMount.azimuthInPosition",
            "MTMount.elevationInPosition",
        ],
        "#3182bd",
        True,
    ),
    ("rotator", ["MTRotator.inPosition"], "#74c476", True),
    ("hexapods (ticks only)", ["MTHexapod.inPosition"], "#17becf", False),
    (
        "dome",
        ["MTDome.azMotion", "MTDomeTrajectory.telescopeVignetted"],
        "#c7a44a",
        True,
    ),
]
TIERS = [
    (0, 9, "bottom"),
    (0, -11, "top"),
    (0, 21, "bottom"),
]  # cycled per cluster of labels


def event_label(name, info):
    """Short tick label for one EFD event."""
    if name == "MTAOS.closedLoopState":
        return AOS_LOOP_STATE.get(int(info["state"]), f"s{int(info['state'])}")
    if name == "ScriptQueue.script":
        return f"{int(info['scriptSalIndex']) % 1000}:{PROCESS_STATE.get(int(info['processState']), '?')}"
    if name == "Script.state":
        return f"{int(info['salIndex']) % 1000}:{SCRIPT_STATE.get(int(info['state']), '?')}"
    if name == "MTCamera.ccsCommandState":
        return f"ccs:{int(info['substate'])}"
    label = EVENT_LABEL.get(name, name)
    if "inPosition" in info:
        label += " T" if info["inPosition"] else " F"
    return label


def plot_gap_critical_path(pick_label, g, events, x_min=-1.5, x_max=None):
    """Two-panel critical-path figure for one gap row `g` with its EFD `events`.

    x_max=None scales the axis to this gap; pass a common x_max to compare several gaps.
    """
    edges = np.concatenate([[0.0], np.cumsum([g[c] for c in SUBCOMPONENTS])])
    gap_len = g.visit_gap
    if x_max is None:
        x_max = gap_len + max(2.0, 0.10 * gap_len)
    x_span = x_max - x_min

    fig = plt.figure(figsize=(13, 11.5), layout="constrained")
    gs = fig.add_gridspec(2, 1, height_ratios=[1.45, 1.0])
    ax_top = fig.add_subplot(gs[0])
    ax_bot = fig.add_subplot(gs[1], sharex=ax_top)

    # --- TOP: the segment staircase (critical path) ---
    n_seg = len(SUBCOMPONENTS)
    for i, c in enumerate(SUBCOMPONENTS):
        y = n_seg - 1 - i  # first segment on the top row
        dur = g[c]
        _name, color = SEGMENTS[c]
        if dur > 0.005:
            ax_top.barh(
                y, dur, left=edges[i], height=0.62, color=color, edgecolor="k", lw=0.5
            )
            label = f"{dur:.2f} s"
            if dur > 0.15 * x_span:  # wide enough for the label to sit inside
                ax_top.text(
                    edges[i] + dur / 2, y, label, ha="center", va="center", fontsize=9
                )
            else:
                ax_top.text(
                    min(edges[i + 1] + 0.008 * x_span, x_max - 0.1),
                    y,
                    label,
                    ha="left",
                    va="center",
                    fontsize=9,
                    clip_on=True,
                )
        else:
            ax_top.plot(edges[i], y, "|", color="#999999", markersize=10)
            ax_top.text(
                edges[i] + 0.008 * x_span,
                y,
                "0 s",
                ha="left",
                va="center",
                fontsize=8,
                color="#888888",
            )
        if i < n_seg - 1:  # dotted hand-off connector down to the next segment
            ax_top.plot(
                [edges[i + 1]] * 2,
                [y - 0.31, y - 1 + 0.31],
                color="#666666",
                lw=0.9,
                ls=":",
            )
    ax_top.set_yticks(range(n_seg))
    ax_top.set_yticklabels(
        [SEGMENTS[c][0] for c in reversed(SUBCOMPONENTS)], fontsize=9
    )
    ax_top.set_ylim(-0.7, n_seg - 0.3)
    ax_top.tick_params(labelbottom=False)
    ax_top.grid(axis="x", alpha=0.3)
    ax_top.set_title(
        f"Deep dive, {pick_label}: gap before visit {g.visit_id} "
        f"({gap_len:.1f} s, {g.observation_reason}, band {g.prev_band}->{g.band})\n"
        "Top: segments tiling the gap (critical path). Bottom: raw EFD events.",
        loc="left",
        fontsize=10,
    )

    # --- BOTTOM: the raw events; at most 3 labels per cluster (2% of the axis span). ---
    for y, (_row_label, names, color, label_events) in enumerate(ROW_STREAMS):
        ts_list = sorted(
            ((ts - g.t_close).total_seconds(), name, info)
            for ts, name, info in events
            if name in names
        )
        cluster_x, tier_i = None, 0
        for x, name, info in ts_list:
            ax_bot.plot(x, y, "|", markersize=15, color=color, markeredgewidth=2)
            if not label_events:
                continue
            if cluster_x is None or x - cluster_x >= 0.02 * x_span:
                cluster_x, tier_i = x, 0
            if tier_i >= len(TIERS):
                continue
            dx, dy, va = TIERS[tier_i]
            ax_bot.annotate(
                event_label(name, info),
                (x, y),
                textcoords="offset points",
                xytext=(dx, dy),
                ha="center",
                va=va,
                fontsize=7.5,
                color=color,
                clip_on=True,
            )
            tier_i += 1
    ax_bot.set_yticks(range(len(ROW_STREAMS)))
    ax_bot.set_yticklabels([r[0] for r in ROW_STREAMS], fontsize=9)
    ax_bot.set_ylim(-0.8, len(ROW_STREAMS) - 0.2)
    ax_bot.set_xlabel("Seconds since shutter close of the previous visit")
    ax_bot.grid(axis="x", alpha=0.3)

    for ax in (ax_top, ax_bot):
        ax.axvline(0, color="k", lw=1.0, alpha=0.6)
        ax.axvline(gap_len, color="k", lw=1.0, alpha=0.6)
    ax_top.set_xlim(x_min, x_max)


COMMON_TIME_AXIS = False  # True: put all examples on one x axis
if COMMON_TIME_AXIS:
    longest = max(g.visit_gap for g in deep_gaps.values())
    common_x_max = longest + max(2.0, 0.10 * longest)
else:
    common_x_max = None
for pick_label, g in deep_gaps.items():
    plot_gap_critical_path(
        pick_label, g, deep_events_by_pick[pick_label], x_max=common_x_max
    )

In [ ]:
# Inspect one visit: draw the gap before it and the gap after it (set inspect_visit_id in the
# parameters cell). Works for any survey gap, including gaps outside the analysis sample.
if inspect_visit_id is None:
    print(
        "inspect_visit_id is None: set it in the parameters cell to inspect a specific visit."
    )
else:
    _vid = int(inspect_visit_id)
    _sides = [
        (f"visit {_vid}, preceding gap", gaps[gaps.visit_id == _vid]),
        (f"visit {_vid}, following gap", gaps[gaps.prev_visit_id == _vid]),
    ]
    if all(len(df) == 0 for _, df in _sides):
        _known = bool((visits.visit_id == _vid).any())
        raise ValueError(
            f"visit {_vid} has no survey gap on either side"
            + (
                " (it is in the range, but neither it nor the image after it is a survey visit)"
                if _known
                else f" -- it is not among the {len(visits)} images of {range_label}; "
                "check the visit_id, or widen day_obs / day_obs_end"
            )
        )
    for _label, _df in _sides:
        if len(_df) == 0:
            print(
                f"{_label}: no survey visit on that side "
                "(first or last survey visit of the night, or the neighbour is not a survey visit)."
            )
            continue
        _g = _df.iloc[0]
        if pd.isna(_g.t_close) or pd.isna(_g.t_open):
            print(
                f"{_label}: shutter events could not be matched (class {_g.gap_class}); "
                "the gap has no window to decompose."
            )
            continue
        _in_sample = (
            _g.gap_class in SAMPLE_CLASSES
            and _g.stratum in SAMPLE_STRATA
            and _g.day_obs in NIGHTS_IN_SAMPLE
        )
        _note = "" if _in_sample else "   [NOT in the analysis sample]"
        print(
            f"{_label}: before visit {_g.visit_id}, dayObs {_g.day_obs}, {_g.visit_gap:.2f} s, "
            f"{_g.stratum}, class {_g.gap_class}, band {_g.prev_band}->{_g.band}{_note}"
        )
        _events = fetch_gap_events(_g)
        if verbose_output:
            for _ts, _name, _info in _events:
                _dt = (_ts - _g.t_close).total_seconds()
                vprint(
                    f"{_dt:+8.2f} s  {_name:34s} "
                    + ", ".join(f"{k}={v}" for k, v in _info.items())
                )
            vprint()
        plot_gap_critical_path(
            _label + ("" if _in_sample else "  [not in sample]"), _g, _events
        )

## Statistics

Per-segment time per visit over the analysis sample, with all nights pooled:

- **median** and **mean**, reported separately. The median describes a routine visit; the mean includes the long tail and drives the integrated cost.
- **min** (over gaps where the segment occurred) and **max**.
- **present_in**: how often the segment occurred.
- **total per night** and **share of gap time**.

After the main table come splits by stratum and class, by limiting axis for Mount slew+settle, and AOS statistics for gaps where a correction was issued.

In [ ]:
# Per-segment table over the analysis sample. Median, mean, total and share count every gap (a
# segment that did not occur is 0 s, so they add up); min and present_in count only gaps where the
# segment lasted more than PRESENT_S. min is shown to 3 decimals.
MIN_DECIMALS = 3
sample = analysis_sample()


def component_stats(sample, cols):
    vals = sample[cols]
    present = vals > PRESENT_S
    tab = pd.DataFrame(
        {
            "median": vals.median(),
            "mean": vals.mean(),
            "min": vals.where(present).min(),
            "max": vals.max(),
            "present_in": present.mean(),
            "total_min_per_night": vals.sum() / 60 / N_NIGHTS,
        }
    )
    tab["share_of_gap_time"] = vals.sum() / sample.visit_gap.sum()
    return tab


def round_for_display(tab):
    """2 decimals everywhere except min (MIN_DECIMALS)."""
    out = tab.round(2)
    if "min" in out.columns:
        out["min"] = tab["min"].round(MIN_DECIMALS)
    return out


print(
    f"Analysis sample, {range_label}: {len(sample)} gaps over {N_NIGHTS} night{'s' if N_NIGHTS > 1 else ''}; "
    f"by stratum {sample.stratum.value_counts().to_dict()}, by class {sample.gap_class.value_counts().to_dict()}"
)

mech_tab = component_stats(sample, MECH_COMPONENTS + ["visit_gap"]).rename(
    index={"visit_gap": "TOTAL gap"}
)
print(
    "Segments (seconds). min and present_in count only gaps where the segment lasted more than "
    f"PRESENT_S = {PRESENT_S:g} s."
)
display(round_for_display(mech_tab))

print("Median / mean gap (s) by stratum and class:")
display(
    sample.groupby(["stratum", "gap_class"])
    .visit_gap.agg(["size", "median", "mean"])
    .round(2)
)

# Mount slew+settle by limiting axis (rows sum to the mount segment; min is over all gaps).
print("\nMount slew+settle by limiting axis (seconds):")
display(
    sample.groupby("slew_limited_by")
    .c_slew_mount.agg(
        n="size",
        median="median",
        mean="mean",
        min="min",
        max="max",
        total_min_per_night=lambda s: s.sum() / 60 / N_NIGHTS,
    )
    .round(2)
)
moved = sample[sample.t_motion.notna()]
print(
    f"  slew start latency (raDecTarget -> first axis moves): median {moved.c_slew_latency.median():.2f} s"
)
print(
    f"  each axis on its own (overlapping, not additive): "
    f"azimuth median {moved.az_axis_s.median():.2f} s / mean {moved.az_axis_s.mean():.2f} s, "
    f"elevation median {moved.el_axis_s.median():.2f} s / mean {moved.el_axis_s.mean():.2f} s"
)
no_mount_motion = sample[sample.t_motion.isna()]
if len(no_mount_motion):
    _ids = [str(v) for v in no_mount_motion.visit_id]
    print(
        f"  gaps where the mount did not move: "
        f"{len(no_mount_motion)} ({', '.join(_ids[:10])}{', ...' if len(_ids) > 10 else ''})"
    )

# AOS, for gaps where each kind of correction was issued.
pre = sample[sample.t_aos_pre.notna()]
post = sample[sample.t_post_flag.notna()]
none = sample[sample.t_aos_pre.isna() & sample.t_post_flag.isna()]
print(
    f"\nAOS Correction (pre-slew) on {len(pre)} of {len(sample)} gaps ({len(pre) / len(sample):.0%}): "
    f"median {pre.c_aos_pre.median():.2f} s, mean {pre.c_aos_pre.mean():.2f} s; "
    f"gap median {pre.visit_gap.median():.2f} s"
)
print(
    f"AOS Correction (post-slew) cycle on {len(post)} of {len(sample)} gaps ({len(post) / len(sample):.0%}); "
    f"gap median {post.visit_gap.median():.2f} s. Within those gaps (median / mean, s):"
)
for col in ["c_inpos", "c_aos_post", "c_aos_pointing", "c_dispatch"]:
    print(
        f"  {SEGMENT_NAME[col]:28s} {post[col].median():5.2f} / {post[col].mean():5.2f}"
    )
print(f"No AOS correction: gap median {none.visit_gap.median():.2f} s")

In [ ]:
# Distributions: one histogram per mechanism segment on a LOG count scale, so the tails are
# visible (on a linear scale the peak hides them). All panels share the same bins for
# comparability; values beyond the last bin are clipped into it (the panel title says how many).
# In a multi-night run these pool every night in the sample (night-to-night changes are in the
# next section).
HIST_MAX_S = 40.0
bins = np.arange(0, HIST_MAX_S + 0.5, 0.5)

fig, axes = plt.subplots(4, 4, figsize=(20, 15), sharex=True, sharey=True)
axes = axes.ravel()
for ax, comp in zip(axes, MECH_COMPONENTS):
    vals = sample[comp]
    n_over = int((vals > HIST_MAX_S).sum())
    ax.hist(
        vals.clip(upper=HIST_MAX_S), bins=bins, color=SEGMENT_COLOR[comp], alpha=0.85
    )
    ax.set_yscale("log")
    ax.axvline(
        vals.median(),
        color="k",
        linestyle="--",
        lw=1.2,
        label=f"median {vals.median():.1f} s",
    )
    ax.axvline(
        vals.mean(), color="k", linestyle=":", lw=1.2, label=f"mean {vals.mean():.1f} s"
    )
    ax.set_title(
        comp + (f"  ({n_over} above {HIST_MAX_S:.0f} s in last bin)" if n_over else ""),
        fontsize=11,
    )
    ax.legend(fontsize=9, loc="upper right")
    ax.grid(alpha=0.3, which="both")
    ax.set_xlabel("seconds")
    ax.set_ylabel("gaps (log scale)")
# One extra panel after the segments: the total gap, same axes, for reference; hide the rest.
ax = axes[len(MECH_COMPONENTS)]
vals = sample.visit_gap
n_over = int((vals > HIST_MAX_S).sum())
ax.hist(vals.clip(upper=HIST_MAX_S), bins=bins, color="#555555", alpha=0.85)
ax.set_yscale("log")
ax.axvline(
    vals.median(),
    color="k",
    linestyle="--",
    lw=1.2,
    label=f"median {vals.median():.1f} s",
)
ax.axvline(
    vals.mean(), color="k", linestyle=":", lw=1.2, label=f"mean {vals.mean():.1f} s"
)
ax.set_title(
    "TOTAL gap (for reference)"
    + (f"  ({n_over} above {HIST_MAX_S:.0f} s)" if n_over else ""),
    fontsize=11,
)
ax.legend(fontsize=9, loc="upper right")
ax.grid(alpha=0.3, which="both")
ax.set_xlabel("seconds")
for ax in axes[len(MECH_COMPONENTS) + 1 :]:
    ax.set_visible(False)

fig.suptitle(
    f"Mechanism-segment distributions over {len(sample)} gaps in the analysis sample, {range_label}"
    + (f" ({N_NIGHTS} nights pooled)" if multi_night else "")
    + " (log count scale; dashed = median, dotted = mean)",
    y=0.995,
    fontsize=12,
)
plt.tight_layout()

In [ ]:
# The whole night at a glance (single-night runs): each gap in the analysis sample as a stacked bar
# of its mechanism segments, in time order. Structure to look for: the pre-slew AOS correction
# segment, the occasional post-slew AOS cycle (correction + pointing correction + dispatch),
# dome/rotator spikes, long mount slews, and the steady orchestration floor.
# A multi-night run has thousands of gaps, too many for one bar each; its per-night composition
# is in the night-to-night comparison below.
if multi_night:
    print(
        f"{len(sample)} gaps over {N_NIGHTS} nights: the gap-by-gap plot is drawn for single-night runs only; "
        "see the night-to-night comparison below for the per-night composition."
    )
else:
    plot_sample = sample.sort_values("obs_start_mjd")
    x = np.arange(len(plot_sample))
    bottom = np.zeros(len(plot_sample))

    fig, ax = plt.subplots(figsize=(15, 4.5))
    for comp in MECH_COMPONENTS:
        ax.bar(
            x,
            plot_sample[comp],
            bottom=bottom,
            width=1.0,
            color=SEGMENT_COLOR[comp],
            label=comp,
        )
        bottom += plot_sample[comp].values
    ax.set_xlim(-1, len(plot_sample))
    ax.set_ylim(0, min(60, bottom.max() * 1.05))
    ax.set_xlabel("Gaps in the analysis sample, in time order through the night")
    ax.set_ylabel("Seconds")
    ax.legend(loc="upper right", ncol=7, fontsize=8)
    ax.set_title(
        f"Every visit gap in the analysis sample of {range_label}, decomposed into mechanism segments",
        loc="left",
    )
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()

In [ ]:
# Set-aside gaps (filter changes, gaps after a non-survey image), reported but not decomposed,
# and the log of gaps removed from the analysis (non_operating, no_shutter_event).
def gap_summary(sample, label):
    if len(sample) == 0:
        print(f"{label:36s} none")
        return
    print(
        f"{label:36s} n={len(sample):5d}  median {sample.visit_gap.median():7.1f} s  "
        f"mean {sample.visit_gap.mean():7.1f} s  total {sample.visit_gap.sum() / 60:7.1f} min"
    )


in_nights = gaps[gaps.day_obs.isin(NIGHTS_IN_SAMPLE)]
in_strata = in_nights[in_nights.stratum.isin(SAMPLE_STRATA)]
print(
    f"Set-aside gap samples, {range_label}, {N_NIGHTS} night(s) in the sample (strata {SAMPLE_STRATA}):\n"
)
gap_summary(in_strata.query("gap_class == 'filter_change'"), "filter change")
gap_summary(
    in_strata.query("gap_class == 'after_non_survey'"), "after non-survey image"
)

n_excl = in_nights.gap_class.isin(EXCLUDED_CLASSES).sum()
print(
    f"\nRemoved from the analysis: {int(n_excl)} gaps "
    f"({', '.join(f'{int((in_nights.gap_class == c).sum())} {c}' for c in EXCLUDED_CLASSES)})"
)
_causes = in_nights.loc[in_nights.gap_class == "non_operating", "exclude_cause"]
if len(_causes):
    print(
        "  by cause: "
        + ", ".join(f"{int(n)} {c}" for c, n in _causes.value_counts().items())
    )

# Filter changes: duration of the exchange itself.
fc = in_strata.query("gap_class == 'filter_change'").copy()
if len(fc) > 0:
    fc["t_fc_start"] = first_in_window(ev_filter_start, fc.t_close, fc.t_open)
    fc["t_fc_end"] = last_in_window(ev_filter_end, fc.t_close, fc.t_open)
    fc["filter_exchange_s"] = (fc.t_fc_end - fc.t_fc_start).dt.total_seconds()
    found = fc[fc.filter_exchange_s.notna() & (fc.filter_exchange_s > 0)]
    print(
        f"\nFilter-change gaps with start/endSetFilter found: {len(found)} of {len(fc)}"
    )
    if len(found) > 0:
        print(
            f"  exchange (startSetFilter -> endSetFilter): median {found.filter_exchange_s.median():.1f} s"
        )
        print(
            f"  rest of those gaps (handover + two slews):  median "
            f"{(found.visit_gap - found.filter_exchange_s).median():.1f} s"
        )

## Night-to-night comparison (ranges only)

Each point or bar is one night in `NIGHTS_IN_SAMPLE`.

1. **Gap per night:** median, mean and interquartile range, with the number of gaps per night.
2. **Mean gap by segment:** per-night segment means, stacked. Means add up, so the height of each stack is the night's mean gap.
3. **Each segment over time:** one panel per segment (each with its own y scale), plus the AOS correction rates and how often the slew was limited by azimuth.

The x axis is **ordinal**: one position per night. Stretches of `NIGHT_BREAK_DAYS` or more nights with no data (such as shutdowns) are removed, so distance along the axis counts nights with data, not elapsed time. Set `COMPRESS_NIGHT_AXIS = False` for a calendar axis.

In [ ]:
# Per-night aggregates on an ordinal night axis, and figures 1-2. Runs of NIGHT_BREAK_DAYS or more
# nights with no data are collapsed out of the axis (marked "//" on the x axis); shorter holes stay
# as blanks. Each stretch of data starts with a dated tick, so the jump across a break is visible.
from matplotlib.transforms import blended_transform_factory

COMPRESS_NIGHT_AXIS = True  # collapse long stretches without data (e.g. a shutdown)
NIGHT_BREAK_DAYS = 7  # a run of this many consecutive nights with no data is collapsed
SHOW_NIGHT_BREAK_MARKS = (
    False  # also draw an annotated vertical line at each collapsed stretch
)

_has_data = pd.Index(DAY_OBS_LIST).isin(NIGHTS_IN_SAMPLE)
_keep = np.ones(len(DAY_OBS_LIST), dtype=bool)
_runs = []  # (first index, last index + 1) of each collapsed stretch
if COMPRESS_NIGHT_AXIS:
    _i = 0
    while _i < len(_keep):
        if _has_data[_i]:
            _i += 1
            continue
        _j = _i
        while _j < len(_keep) and not _has_data[_j]:
            _j += 1
        if _j - _i >= NIGHT_BREAK_DAYS:
            _keep[_i:_j] = False
            _runs.append((_i, _j))
        _i = _j

NIGHTS_PLOT = [
    d for d, k in zip(DAY_OBS_LIST, _keep) if k
]  # the nights drawn, in order
NIGHT_X = np.arange(len(NIGHTS_PLOT), dtype=float)  # x position on the ordinal axis
NIGHT_LABELS = [f"{str(d)[:4]}-{str(d)[4:6]}-{str(d)[6:]}" for d in NIGHTS_PLOT]
# Position of each collapsed stretch: halfway between the kept nights on either side.
_cum = np.cumsum(_keep)
NIGHT_GAP_MARKS = [
    (float(_cum[i]) - 0.5, j - i, DAY_OBS_LIST[i], DAY_OBS_LIST[j - 1])
    for i, j in _runs
]
# Index of the first drawn night after each break inside the axis (not at either end).
NIGHT_BREAK_IDX = sorted(
    {int(x + 0.5) for x, *_ in NIGHT_GAP_MARKS if 0 < x + 0.5 < len(NIGHTS_PLOT)}
)
if NIGHT_GAP_MARKS:
    print(
        f"Night axis: {len(NIGHTS_PLOT)} of {n_nights_range} nights drawn; "
        f"{len(NIGHT_GAP_MARKS)} stretch(es) without data collapsed: "
        + ", ".join(f"{d0}-{d1} ({n} nights)" for _x, n, d0, d1 in NIGHT_GAP_MARKS)
        + "."
    )


def _night_ticks(max_ticks):
    """Tick positions: the first night of each stretch, then every `step` nights within it."""
    n = len(NIGHT_X)
    step = max(1, int(np.ceil(n / max(1, max_ticks))))
    starts = [0] + NIGHT_BREAK_IDX
    ends = NIGHT_BREAK_IDX + [n]
    ticks = []
    for s, e in zip(starts, ends):
        while ticks and s - ticks[-1] < max(
            1, step / 2
        ):  # make room for the stretch's first tick
            ticks.pop()
        ticks.extend(range(s, e, step))
    return ticks


def format_night_axis(ax, label_breaks=False, max_ticks=10):
    """Ordinal night axis: dated ticks that restart at each break, and a "//" mark at each break."""
    ax.set_xlim(-0.8, len(NIGHT_X) - 0.2)
    ticks = _night_ticks(max_ticks)
    ax.set_xticks(ticks)
    ax.set_xticklabels(
        [NIGHT_LABELS[t] for t in ticks],
        fontsize=9 if max_ticks > 5 else 8,
        rotation=30,
        ha="right",
    )
    trans = blended_transform_factory(ax.transData, ax.transAxes)
    for idx in NIGHT_BREAK_IDX:
        ax.text(
            idx - 0.5, 0, "//", transform=trans, ha="center", va="center",
            fontsize=11, fontweight="bold", color="#444444", clip_on=False,
            bbox=dict(facecolor="white", edgecolor="none", pad=0.5),
        )  # fmt: skip
    if not SHOW_NIGHT_BREAK_MARKS:
        return
    for x, n_skipped, d0, d1 in NIGHT_GAP_MARKS:
        ax.axvline(x, color="#b03a2e", ls=(0, (4, 3)), lw=1.3, zorder=0.5)
        if label_breaks:
            ax.annotate(
                f"{n_skipped} nights without data\n{d0} - {d1}",
                (x, 0.0),
                xycoords=("data", "axes fraction"),
                textcoords="offset points",
                xytext=(0, 5),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="#b03a2e",
            )


if not multi_night:
    print(
        "Single-night run: the night-to-night comparison needs a range of nights (set day_obs_end)."
    )
else:
    _by_night = sample.groupby("day_obs")
    per_night = pd.DataFrame(
        {
            "gaps": _by_night.size(),
            "median_gap": _by_night.visit_gap.median(),
            "mean_gap": _by_night.visit_gap.mean(),
            "p25_gap": _by_night.visit_gap.quantile(0.25),
            "p75_gap": _by_night.visit_gap.quantile(0.75),
            "p90_gap": _by_night.visit_gap.quantile(0.90),
            "aos_pre_rate": sample.t_aos_pre.notna().groupby(sample.day_obs).mean(),
            "aos_post_rate": sample.t_post_flag.notna().groupby(sample.day_obs).mean(),
            "azimuth_limited": (sample.slew_limited_by == "azimuth")
            .groupby(sample.day_obs)
            .mean(),
            "gap_hours": _by_night.visit_gap.sum() / 3600,
            "observable_hours": night_hours_by_night,
        }
    ).reindex(NIGHTS_PLOT)
    per_night.index.name = "day_obs"
    seg_mean_by_night = _by_night[MECH_COMPONENTS].mean().reindex(NIGHTS_PLOT)
    seg_median_by_night = _by_night[MECH_COMPONENTS].median().reindex(NIGHTS_PLOT)
    vdisplay(per_night.dropna(subset=["gaps"]).round(2))
    vdisplay(seg_mean_by_night.dropna(how="all").round(2))

    # --- Figure 1: the gap per night (top) and the gaps behind each night (bottom). ---
    fig, (ax_g, ax_n) = plt.subplots(
        2, 1, figsize=(15, 6.5), sharex=True, height_ratios=[3, 1], layout="constrained"
    )
    ax_g.vlines(
        NIGHT_X,
        per_night.p25_gap,
        per_night.p75_gap,
        color="#9e9e9e",
        lw=4,
        alpha=0.6,
        label="interquartile range",
    )
    ax_g.plot(NIGHT_X, per_night.median_gap, "o", color="#2c5f8a", ms=6, label="median")
    ax_g.plot(
        NIGHT_X,
        per_night.mean_gap,
        "o",
        mfc="none",
        mec="#d94801",
        mew=1.5,
        ms=8,
        label="mean",
    )
    ax_g.axhline(
        sample.visit_gap.median(),
        color="#2c5f8a",
        ls="--",
        lw=1,
        alpha=0.7,
        label=f"median over the range ({sample.visit_gap.median():.1f} s)",
    )
    ax_g.set_ylim(0, None)
    ax_g.set_ylabel("Visit gap per night (s)")
    ax_g.grid(alpha=0.3)
    ax_g.legend(loc="upper right", ncol=4, fontsize=9)
    ax_g.set_title(
        f"Inter-visit gap, night by night: {range_label} ({N_NIGHTS} of {n_nights_range} nights in the sample)",
        loc="left",
    )
    ax_n.bar(NIGHT_X, per_night.gaps, width=0.8, color="#7f7f7f")
    ax_n.set_ylabel("Gaps in sample")
    ax_n.grid(axis="y", alpha=0.3)
    format_night_axis(ax_n, label_breaks=True)

    # --- Figure 2: the per-night MEAN gap, split into its segments (means add up). ---
    fig, ax = plt.subplots(figsize=(15, 6), layout="constrained")
    bottom = np.zeros(len(NIGHT_X))
    for comp in MECH_COMPONENTS:
        v = seg_mean_by_night[comp].fillna(0).values
        ax.bar(
            NIGHT_X,
            v,
            bottom=bottom,
            width=0.8,
            color=SEGMENT_COLOR[comp],
            label=comp,
            edgecolor="white",
            linewidth=0.3,
        )
        bottom += v
    ax.set_ylabel("Mean seconds per gap")
    ax.grid(axis="y", alpha=0.3)
    ax.set_title(
        f"What the mean gap is made of, night by night: {range_label} "
        "(stack height = the night's mean gap; segments in time order from the bottom)",
        loc="left",
    )
    format_night_axis(ax, label_breaks=True)
    ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=9, reverse=True)

In [ ]:
# Figure 3: each segment over time, one panel per segment with its own y scale (so a change in a
# 0.1 s segment is as visible as one in the 5 s mount slew): per-night mean (dots in the segment
# colour, solid line) and median (hollow dots, dashed line). The last panels show how often AOS
# corrections were applied before / after the slew, and how often the slew was azimuth-limited.
# Lines break across nights without data; long stretches without data are collapsed out of the
# axis entirely (see the night-axis note in the previous cell, which also lists them). These
# panels are narrow, so they carry fewer date labels than the wide figures above.
if multi_night:
    fig, axes = plt.subplots(4, 4, figsize=(20, 14), sharex=True, layout="constrained")
    axes = axes.ravel()
    for ax, comp in zip(axes, MECH_COMPONENTS):
        ax.plot(
            NIGHT_X,
            seg_mean_by_night[comp],
            "-o",
            color="#555555",
            lw=1,
            ms=5,
            mfc=SEGMENT_COLOR[comp],
            mec="#333333",
            mew=0.6,
            label="mean",
        )
        ax.plot(
            NIGHT_X,
            seg_median_by_night[comp],
            "--o",
            color="#999999",
            lw=1,
            ms=4,
            mfc="none",
            mec="#555555",
            mew=0.8,
            label="median",
        )
        ax.set_ylim(0, None)
        ax.set_title(comp, fontsize=11, loc="left")
        ax.set_ylabel("seconds per gap")
        ax.grid(alpha=0.3)
    axes[0].legend(fontsize=9, loc="best")

    # AOS correction rates (pre- and post-slew), as a share of the night's gaps.
    ax = axes[len(MECH_COMPONENTS)]
    ax.plot(
        NIGHT_X,
        100 * per_night.aos_pre_rate,
        "-o",
        color=SEGMENT_COLOR["AOS Correction (pre-slew)"],
        lw=1.2,
        ms=5,
        label="pre-slew",
    )
    ax.plot(
        NIGHT_X,
        100 * per_night.aos_post_rate,
        "-s",
        color=SEGMENT_COLOR["AOS Correction (post-slew)"],
        lw=1.2,
        ms=5,
        label="post-slew",
    )
    ax.set_ylim(0, None)
    ax.set_title("AOS corrections applied", fontsize=11, loc="left")
    ax.set_ylabel("% of gaps")
    ax.legend(fontsize=9, loc="best")
    ax.grid(alpha=0.3)

    # Mount sub-category: share of gaps whose slew was limited by azimuth (the rest: elevation).
    ax = axes[len(MECH_COMPONENTS) + 1]
    ax.plot(
        NIGHT_X,
        100 * per_night.azimuth_limited,
        "-o",
        color=SEGMENT_COLOR["Mount slew+settle"],
        lw=1.2,
        ms=5,
    )
    ax.set_ylim(0, 100)
    ax.set_title("Mount slew limited by azimuth", fontsize=11, loc="left")
    ax.set_ylabel("% of gaps")
    ax.grid(alpha=0.3)

    for ax in axes[len(MECH_COMPONENTS) + 2 :]:
        ax.set_visible(False)
    for ax in axes:
        if ax.get_visible():
            format_night_axis(ax, max_ticks=4)
            ax.tick_params(labelbottom=True)
    fig.suptitle(
        f"Mechanism segments night by night, {range_label} ({N_NIGHTS} nights; "
        "per-night mean = filled dots, median = hollow dots; each panel has its own y scale)",
        fontsize=12,
    )

## Results

1. **Headline box:** median, mean, min and max per visit for each of the thirteen segments, how often each occurred, and its share of gap time.
2. **Ranked contributors:** the thirteen segments plus whole filter-change gaps, ranked by total time and extrapolated to a nights-per-year cost.

$$\text{nights-per-year cost} = \frac{\text{segment minutes over the sampled nights}}{\text{observable minutes of those nights}} \times \texttt{EXTRAPOLATION\_NIGHTS}$$

In [ ]:
# HEADLINE: the visit gap and its thirteen segments.
from IPython.display import HTML, display

no_aos = sample[sample.t_aos_pre.isna() & sample.t_post_flag.isna()]
TD = "<td style='text-align: right; padding: 2px 0 2px 14px;'>"


def stat_cells(vals):
    present = vals[vals > PRESENT_S]
    return (
        f"{TD}<b>{vals.median():.2f} s</b></td>{TD}{vals.mean():.2f} s</td>"
        f"{TD}{present.min() if len(present) else float('nan'):.2f} s</td>{TD}{vals.max():.2f} s</td>"
        f"{TD}{len(present) / len(vals):.0%}</td>"
    )


def rows_html(comps):
    return "".join(
        f"<tr><td style='padding: 2px 12px 2px 0;'>{c}</td>{stat_cells(sample[c])}"
        f"{TD}{sample[c].sum() / sample.visit_gap.sum():.0%}</td></tr>"
        for c in comps
    )


header_row = (
    "<tr><th style='text-align: left;'>&nbsp;</th>"
    + "".join(
        f"<th style='text-align: right; padding-left: 14px;'>{label}</th>"
        for label in ["median", "mean", "min", "max", "present in", "share"]
    )
    + "</tr>"
)
total_row = (
    f"<tr style='border-top: 1px solid #999;'><td style='padding: 2px 12px 2px 0;'><b>TOTAL gap</b></td>"
    f"{stat_cells(sample.visit_gap)}{TD}100%</td></tr>"
)
strata_txt = ", ".join(f"{n} {s}" for s, n in sample.stratum.value_counts().items())
az_share = (sample.slew_limited_by == "azimuth").mean()
nights_txt = f", {N_NIGHTS} of {n_nights_range} nights with data" if multi_night else ""
night_spread_li = ""
if multi_night:
    _med = sample.groupby("day_obs").visit_gap.median()
    night_spread_li = (
        f"<li>Per-night median gap ranged from {_med.min():.1f} s ({_med.idxmin()}) to "
        f"{_med.max():.1f} s ({_med.idxmax()}).</li>"
    )

display(
    HTML(
        f"""
<div style="border: 3px solid #2c5f8a; border-radius: 8px; padding: 14px 20px; margin: 10px 0; max-width: 980px;
            background: rgba(128, 128, 128, 0.08); font-family: sans-serif;">
  <div style="font-size: 1.05em; font-weight: bold; letter-spacing: 0.04em;">
    ROSP-23 INTER-VISIT INTERVAL BREAKDOWN &mdash; {range_label}{nights_txt}, {len(sample)} visit gaps ({strata_txt})
  </div>
  <div style="font-size: 2.6em; font-weight: bold; color: #2c5f8a; margin: 6px 0 0 0;">
    {sample.visit_gap.median():.1f} s median &nbsp;/&nbsp; {sample.visit_gap.mean():.1f} s mean
  </div>
  <div style="margin: 10px 0;">
    <div style="font-weight: bold; margin-bottom: 4px;">Segments per visit gap (in time order)</div>
    <table style="border-collapse: collapse;">{header_row}{rows_html(MECH_COMPONENTS)}{total_row}</table>
  </div>
  <div style="font-weight: bold;">Notes</div>
  <ul style="margin: 2px 0 0 0;">
    <li>Gaps run from EFD endShutterClose to startShutterOpen; the segments sum exactly to each gap.</li>
    {night_spread_li}
    <li><b>min</b> and <b>present in</b> count only gaps where the segment occurred. Median, mean and share
        count every gap, so optional segments can have a 0 s median. Segment medians do not add up to the
        median gap.</li>
    <li>Mount slew+settle was limited by azimuth on {az_share:.0%} of gaps.</li>
    <li>AOS corrections were applied before the slew on {sample.t_aos_pre.notna().mean():.0%} of gaps
        and after it on {sample.t_post_flag.notna().mean():.0%}. Gaps with neither have a median of
        {no_aos.visit_gap.median():.1f} s.</li>
    <li>Filter changes and gaps after non-survey images are reported separately. Gaps affected by
        FAULT / IDLE / DOWNTIME are excluded.</li>
  </ul>
</div>
"""
    )
)

In [ ]:
# Ranked contributors (thirteen segments + whole filter-change gaps) by total time, with the
# nights-per-year extrapolation over the sampled nights.
EXTRAPOLATION_NIGHTS = 205  # observing nights per year when discouting weather (37.5%) and availability (10%) from OSS-REQ-0080

contributors = {
    c: sample[c].sum() / 60 for c in MECH_COMPONENTS
}  # minutes over the sample
contributors["Filter changes (whole gaps)"] = (
    gaps[
        gaps.stratum.isin(SAMPLE_STRATA)
        & (gaps.gap_class == "filter_change")
        & gaps.day_obs.isin(NIGHTS_IN_SAMPLE)
    ].visit_gap.sum()
    / 60
)

observable_minutes = night_hours_by_night.loc[NIGHTS_IN_SAMPLE].sum() * 60
ranked = pd.DataFrame({"minutes_total": pd.Series(contributors)}).sort_values(
    "minutes_total", ascending=False
)
ranked["minutes_per_night"] = ranked.minutes_total / N_NIGHTS
ranked["share_of_observable_night"] = ranked.minutes_total / observable_minutes
ranked["nights_per_year_cost"] = ranked.share_of_observable_night * EXTRAPOLATION_NIGHTS
ranked.index.name = "contributor"
display(
    ranked.round(2) if multi_night else ranked.drop(columns="minutes_total").round(2)
)

fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = [SEGMENT_COLOR.get(n, "#777777") for n in ranked.index[::-1]]
ax.barh(ranked.index[::-1], ranked.nights_per_year_cost[::-1], color=bar_colors)
for i, v in enumerate(ranked.nights_per_year_cost[::-1]):
    ax.text(v, i, f" {v:.1f}", va="center", fontsize=9, clip_on=True)
ax.set_xlim(0, ranked.nights_per_year_cost.max() * 1.12)
ax.set_xlabel(
    f"Nights-per-year cost (share of the observable night x {EXTRAPOLATION_NIGHTS} nights)"
)
ax.set_title(
    f"Ranked gap-time contributors, {range_label}"
    + (f", {N_NIGHTS} nights pooled" if multi_night else "")
    + "\n(upper bounds on recoverable time; grey bar = whole filter-change gaps)",
    loc="left",
    fontsize=10,
)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

print(
    f"\nGap time in the analysis sample: {sample.visit_gap.sum() / 60:.1f} min "
    f"({sample.visit_gap.sum() / 60 / N_NIGHTS:.1f} min per night) of {observable_minutes / 60:.1f} h of "
    f"observable night over {N_NIGHTS} night{'s' if N_NIGHTS > 1 else ''} "
    f"({sample.visit_gap.sum() / 60 / observable_minutes:.1%})."
)